<div align="right">
    
#### בס"ד
</div>

<div dir="rtl">

# כריית נתונים בפייתון - מטלה 1
### סט נתונים על סרטים מ-Aa עד AL

   # מגישים:
  #  כפיר זיסו - 322883091
  #  ראובן ולדמן - 318344231
  # GitHub  - https://github.com/KfirZiso/imdb-movie-analysis

</div>

<center>
    <img src="https://m.media-amazon.com/images/I/712BUdRyfPL._AC_UF1000,1000_QL80_.jpg" width="400" style="display: inline-block; margin-right: 10px;">
    <img src="https://www.panoramaaudiovisual.com/wp-content/uploads/2019/06/IMDB.jpg" width="700" style="display: inline-block;">
</center>

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re



## Checking Existing Movies from Wikipedia



In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_films:_A"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36'}

try:
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
    else:
        print(f"Connection error: Status code {response.status_code}")
except Exception as e:
    print(f"Error during connection: {e}")

In [ ]:
movies_names = []

try:
    start_node = soup.find(id="Aa")
    end_node = soup.find(id="Am")

    if start_node:
        current = start_node
        while current:
            current = current.find_next()

            if current == end_node:
                break

            if current.name == 'div' and 'div-col' in current.get('class', []):
                items = current.find_all('li')
                for item in items:
                    link = item.find('a')
                    if link:
                        name = link.get_text().strip()
                        if not name.isdigit() and len(name) > 1:
                            movies_names.append(name)

        df_wikipedia = pd.DataFrame(movies_names, columns=['Movie_Title'])
        display(df_wikipedia)
    else:
        print("Could not find start node 'Aa'.")

except Exception as e:
    print(f"Error during data processing: {e}")

,Movie_Title
0,Aa Ab Laut Chalen
1,Aa Aiduguru
2,Aa Ammayi Gurinchi Meeku Cheppali
3,Aa Bb Kk
4,Aa Chithrashalabham Parannotte
...,...
1383,Alyas Pogi: Birador ng Nueva Ecija
1384,Alyas Pogi 2
1385,Alyas Pogi: Ang Pagbabalik
1386,Alyas Pusa: Ang Taong May 13 Buhay


In [ ]:
wiki_total_count = len(df_wikipedia)

print(f"Total movies found in Wikipedia: {wiki_total_count}")

Total movies found in Wikipedia: 1388


# Creating a DataFrame using IMDB Files

We will now import from imdb the movie dataset, prioritizing titles that are also available on Wikipedia.

In [ ]:
wiki_movie_set = set(df_wikipedia['Movie_Title'].unique())
results_list = []
selected_columns = ['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']

chunks = pd.read_csv('title.basics.tsv', sep='\t', low_memory=False, chunksize=50000)

for chunk in chunks:
    chunk['runtimeMinutes'] = pd.to_numeric(chunk['runtimeMinutes'], errors='coerce')
    chunk['startYear'] = pd.to_numeric(chunk['startYear'], errors='coerce')

    base_filter = (
        (chunk['titleType'] == 'movie') &
        (chunk['runtimeMinutes'] >= 60) &
        (chunk['runtimeMinutes'] <= 300)
    )

    wiki_match = chunk[base_filter & (chunk['primaryTitle'].isin(wiki_movie_set))]
    general_match = chunk[base_filter & (chunk['primaryTitle'].str.contains("^A[a-l]", case=False, na=False))]

    combined_chunk = pd.concat([wiki_match, general_match]).drop_duplicates(subset=['tconst'])

    if not combined_chunk.empty:
        temp_df = combined_chunk[selected_columns].copy()
        temp_df['startYear'] = temp_df['startYear'].fillna(0).astype('int32')
        temp_df['runtimeMinutes'] = temp_df['runtimeMinutes'].fillna(0).astype('int32')

        temp_df['genres'] = temp_df['genres'].replace(r'\N', np.nan)
        temp_df['genres'] = temp_df['genres'].fillna("").astype(str)
        temp_df['genres'] = temp_df['genres'].apply(
            lambda x: [i.strip() for i in x.split(',')] if (x.strip() and x != "") else []
     )
        temp_df['genres'] = temp_df['genres'].apply(lambda l: [i for i in l if i])
        temp_df['genres'] = temp_df['genres'].apply(lambda x: np.nan if not x else x)

        results_list.append(temp_df)

if results_list:
    df_title_basics = pd.concat(results_list).reset_index(drop=True)
    display(df_title_basics)
else:
    print("No matches found.")

,tconst,primaryTitle,startYear,runtimeMinutes,genres
0,tt0002605,The Adventures of Kathlyn,1913,300,[Adventure]
1,tt0008816,Adam Bede,1918,60,"[Crime, Drama]"
2,tt0010953,Algol: Tragedy of Power,1920,99,"[Fantasy, Sci-Fi]"
3,tt0011908,Adventures of Tarzan,1921,73,"[Action, Adventure]"
4,tt0011909,The Affairs of Anatol,1921,117,"[Comedy, Drama]"
...,...,...,...,...,...
9505,tt9876994,Aamako Man,2018,60,"[Drama, Family]"
9506,tt9883042,Aakashaganga II,2019,142,[Horror]
9507,tt9900180,Aavahayami,2017,97,[Mystery]
9508,tt9904552,All for the Money,2019,107,[Comedy]


Total movies processed: 9510


### ⬇️ If we wish to restrict the number of movies imported, adjust the parameters below.

In [ ]:
# target = 5000

# wiki_movie_set = set(df_wikipedia['Movie_Title'].unique())
# results_list = []
# selected_columns = ['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']
# current_total = 0

# chunks = pd.read_csv('title.basics.tsv', sep='\t', low_memory=False, chunksize=50000)

# for chunk in chunks:
#     chunk['runtimeMinutes'] = pd.to_numeric(chunk['runtimeMinutes'], errors='coerce')
#     chunk['startYear'] = pd.to_numeric(chunk['startYear'], errors='coerce')

#     base_filter = ( (chunk['titleType'] == 'movie') & (chunk['runtimeMinutes'] >= 60) & (chunk['runtimeMinutes'] <= 300))

#     wiki_match = chunk[base_filter & (chunk['primaryTitle'].isin(wiki_movie_set))]
#     general_match = chunk[base_filter & (chunk['primaryTitle'].str.contains("^A[a-l]", case=False, na=False))]

#     combined_chunk = pd.concat([wiki_match, general_match]).drop_duplicates(subset=['tconst'])

#     if not combined_chunk.empty:
#         temp_df = combined_chunk[selected_columns].copy()
#         temp_df['startYear'] = temp_df['startYear'].fillna(0).astype('int32')
#         temp_df['runtimeMinutes'] = temp_df['runtimeMinutes'].fillna(0).astype('int32')
#         temp_df['genres'] = temp_df['genres'].replace(r'\N', np.nan)
#         temp_df['genres'] = temp_df['genres'].fillna("").astype(str)

#         temp_df['genres'] = temp_df['genres'].apply(
#             lambda x: [i.strip() for i in x.split(',')] if (x.strip() and x != "") else [] )
#         temp_df['genres'] = temp_df['genres'].apply(lambda l: [i for i in l if i])
#         temp_df['genres'] = temp_df['genres'].apply(lambda x: np.nan if not x else x)

#         results_list.append(temp_df)
#         current_total += len(temp_df)

#     if current_total >= target:
#         break

# if results_list:
#     df_title_basics = pd.concat(results_list).head(target).reset_index(drop=True)
#     display(df_title_basics)
# else:
#     print("No matches found.")

## Import the Actors

In [ ]:
results_list = []
file_name = 'title.principals.tsv'
selected_columns = ['tconst', 'nconst']

required_tconsts = set(df_title_basics['tconst'])

try:
    chunks = pd.read_csv(file_name,  sep='\t',  low_memory=False,  chunksize=100000,  compression='gzip')

    for chunk in chunks:
        chunk = chunk[chunk['category'].isin(['actor', 'actress'])]

        filtered_chunk = chunk[chunk['tconst'].isin(required_tconsts)]

        if not filtered_chunk.empty:
            results_list.append(filtered_chunk)

            found_so_far = pd.concat(results_list)['tconst'].nunique()
            if found_so_far >= len(required_tconsts):
                break

    if results_list:
        df_principals_filtered = pd.concat(results_list)

        df_principals_filtered = df_principals_filtered.sort_values(['tconst', 'ordering'])
        df_principals_filtered = df_principals_filtered.groupby('tconst').head(5)

        df_principals_filtered = df_principals_filtered[selected_columns]
        df_principals_filtered.reset_index(drop=True, inplace=True)

        df_principals_filtered
    else:
        print("No matches found.")

except Exception as e:
    print(f"An error occurred: {e}")
df_principals_filtered

,tconst,nconst
0,tt0002605,nm0931031
1,tt0002605,nm0165134
2,tt0002605,nm0139356
3,tt0002605,nm0571186
4,tt0002605,nm0764346
...,...,...
37304,tt9904552,nm2113994
37305,tt9904552,nm5711066
37306,tt9904552,nm9295372
37307,tt9904552,nm1693834


## Import the Rating

In [ ]:
results_list = []
file_name = 'title.ratings.tsv'
selected_columns = ['tconst', 'averageRating','numVotes']

chunks = pd.read_csv(file_name, sep='\t', low_memory=False, chunksize=50000, usecols=selected_columns)

for chunk in chunks:
    filtered_chunk = pd.merge(chunk, df_title_basics[['tconst']], on='tconst')
    results_list.append(filtered_chunk)

df_ratings_filtered = pd.concat(results_list)
df_ratings_filtered.reset_index(drop=True, inplace=True)

df_ratings_filtered

,tconst,averageRating,numVotes
0,tt0002605,5.4,52
1,tt0004872,6.5,316
2,tt0007617,5.2,158
3,tt0007620,5.3,54
4,tt0007623,6.8,19
...,...,...,...
6780,tt9883042,2.1,316
6781,tt9892546,5.1,56
6782,tt9900180,8.3,14
6783,tt9904552,5.4,32


## We will now consolidate all our data sources into a single, unified DataFrame.

In [ ]:
df_actors_grouped = df_principals_filtered.groupby('tconst')['nconst'].apply(list).reset_index()

df_actors_grouped = df_actors_grouped.rename(columns={'nconst': 'lead_actors_ids'})

df_final = pd.merge(df_title_basics, df_ratings_filtered, on='tconst', how='left')
df_final = pd.merge(df_final, df_actors_grouped, on='tconst', how='left')

df_final.reset_index(drop=True, inplace=True)

df_final['numVotes'] = df_final['numVotes'].astype('Int64')

display(df_final)

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0002605,The Adventures of Kathlyn,1913,300,[Adventure],5.4,52,"[nm0931031, nm0165134, nm0139356, nm0571186, n..."
1,tt0008816,Adam Bede,1918,60,"[Crime, Drama]",NaN,<NA>,"[nm0930154, nm0167087, nm0519307, nm0024706, n..."
2,tt0010953,Algol: Tragedy of Power,1920,99,"[Fantasy, Sci-Fi]",6.3,339,"[nm0417837, nm0332024, nm0772300, nm0707765, n..."
3,tt0011908,Adventures of Tarzan,1921,73,"[Action, Adventure]",5.5,187,"[nm0511104, nm0521120, nm0671501, nm0926386, n..."
4,tt0011909,The Affairs of Anatol,1921,117,"[Comedy, Drama]",6.6,1428,"[nm0717468, nm0841797, nm0223296, nm0199841, n..."
...,...,...,...,...,...,...,...,...
9505,tt9876994,Aamako Man,2018,60,"[Drama, Family]",NaN,<NA>,"[nm10514438, nm10521133, nm10521134, nm1052113..."
9506,tt9883042,Aakashaganga II,2019,142,[Horror],2.1,316,"[nm0471447, nm11618001, nm9355289, nm5465692, ..."
9507,tt9900180,Aavahayami,2017,97,[Mystery],8.3,14,"[nm10531566, nm10533868, nm10531569, nm7993373..."
9508,tt9904552,All for the Money,2019,107,[Comedy],5.4,32,"[nm2113994, nm5711066, nm9295372, nm1693834, n..."


# Wikipedia Data Retrieval (Web Scraping)
# In this section, we implement a **Web Scraping** engine to enrich our dataset with additional details from Wikipedia, such as movie budgets, box office performance, and plot summaries.

In [ ]:
def clean_currency_to_millions(text):
    if pd.isna(text) or text == 'N/A':
        return None
    text = re.sub(r'\[.*?\]', '', text)
    lower_text = text.lower().replace(',', '').replace('$', '')
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", lower_text)
    if not numbers:
        return None
    try:
        value = float(numbers[0])
        if 'million' in lower_text:
            return value
        if value > 10000:
            return value / 1_000_000
        return value
    except:
        return None

def get_movie_data_final(movie_title):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}
    res_data = {'Language': 'N/A', 'Country': 'N/A', 'Budget': None, 'Box_Office': None, 'Plot': 'N/A'}
    formatted_name = str(movie_title).strip().replace(' ', '_')
    url_attempts = [f"https://en.wikipedia.org/wiki/{formatted_name}_(film)", f"https://en.wikipedia.org/wiki/{formatted_name}"]
    soup = None
    for url in url_attempts:
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                temp_soup = BeautifulSoup(r.text, 'html.parser')
                if not temp_soup.find('div', id='noarticletext') and not temp_soup.find('table', id='disambigbox'):
                    soup = temp_soup
                    break
        except:
            continue
    if not soup:
        return pd.Series(res_data)
    try:
        infobox = soup.find('table', class_='infobox')
        if infobox:
            mapping = {'Language': 'Language', 'Country': 'Country', 'Budget': 'Budget', 'Box office': 'Box_Office'}
            for search_word, key in mapping.items():
                row = infobox.find('th', string=lambda s: s and search_word.lower() in s.lower())
                if row:
                    val_node = row.find_next('td')
                    if val_node:
                        val = val_node.get_text(separator=" ").strip()
                        if key in ['Budget', 'Box_Office']:
                            res_data[key] = clean_currency_to_millions(val)
                        elif key == 'Language':
                            clean_lang = re.sub(r'\[.*?\]', '', val).strip()
                            res_data[key] = clean_lang.split()[0] if clean_lang else 'N/A'
                        else:
                            res_data[key] = re.sub(r'\[.*?\]', '', val).strip()
        plot_node = None
        for h2 in soup.find_all('h2'):
            if any(word in h2.get_text().lower() for word in ['plot', 'summary', 'synopsis', 'story']):
                plot_node = h2
                break
        if plot_node:
            paragraphs = []
            for el in plot_node.find_all_next():
                if el.name == 'h2': break
                if el.name == 'p':
                    txt = re.sub(r'\[.*?\]', '', el.get_text().strip())
                    if len(txt) > 20: paragraphs.append(txt)
            if paragraphs:
                res_data['Plot'] = " ".join(dict.fromkeys(paragraphs))
        return pd.Series(res_data)
    except:
        return pd.Series(res_data)

wiki_results = df_final['primaryTitle'].apply(get_movie_data_final)
df_final[['Language', 'Country', 'Budget', 'Box_Office', 'Plot']] = wiki_results
df_final['Budget'] = pd.to_numeric(df_final['Budget'], errors='coerce')
df_final['Box_Office'] = pd.to_numeric(df_final['Box_Office'], errors='coerce')
display(df_final.head(10))

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,Box_Office,Plot
0,tt0002605,The Adventures of Kathlyn,1913,300,[Adventure],5.4,52,"[nm0931031, nm0165134, nm0139356, nm0571186, n...",N/A,United States,NaN,NaN,N/A
1,tt0008816,Adam Bede,1918,60,"[Crime, Drama]",NaN,<NA>,"[nm0930154, nm0167087, nm0519307, nm0024706, n...",Silent,United Kingdom,NaN,NaN,A squire's grandson saves a farmer's niece fro...
2,tt0010953,Algol: Tragedy of Power,1920,99,"[Fantasy, Sci-Fi]",6.3,339,"[nm0417837, nm0332024, nm0772300, nm0707765, n...",Silent,Weimar Republic,NaN,NaN,"The story follows the life of Robert Herne, wh..."
3,tt0011908,Adventures of Tarzan,1921,73,"[Action, Adventure]",5.5,187,"[nm0511104, nm0521120, nm0671501, nm0926386, n...",Hindi,N/A,NaN,NaN,This is the popular story of Tarzan retold in ...
4,tt0011909,The Affairs of Anatol,1921,117,"[Comedy, Drama]",6.6,1428,"[nm0717468, nm0841797, nm0223296, nm0199841, n...",Silent,United States,0.176508,1.2,"Socialite Anatol Spencer, finding his relation..."
5,tt0011912,After the Show,1921,60,"[Drama, Romance]",6.2,16,"[nm0392442, nm0497759, nm0644728, nm0816123, n...",Silent,United States,NaN,NaN,"As described in a film magazine, country girl ..."
6,tt0013822,Alice Adams,1923,73,[Drama],4.2,19,"[nm0896538, nm0319257, nm0329467, nm0574739, n...",N/A,N/A,NaN,NaN,N/A
7,tt0015532,The Adventures of Prince Achmed,1926,80,"[Adventure, Animation, Drama]",7.8,7700,NaN,Silent,Germany,NaN,100.0,An African sorcerer conjures up a flying horse...
8,tt0016575,Across the Pacific,1926,78,"[Adventure, Romance, War]",6.4,41,"[nm0089524, nm0936079, nm0001485, nm0828314, n...",English,United States,0.576000,1.3,"On November 17, 1941, on Governor's Island in ..."
9,tt0017590,An Affair of the Follies,1927,70,"[Drama, Romance]",NaN,<NA>,"[nm0832011, nm0235521, nm0400763, nm0831736, n...",Silent,United States,NaN,NaN,N/A


Final Dataset Integration
We have successfully integrated the IMDB foundational data with the enriched Wikipedia metrics. The final dataset now includes structured information for each movie.

In [ ]:
print(f"Number of missing plots: {len(df_final[df_final['Plot'] == 'N/A'])}")

Number of missing plots: 7483


To maximize our dataset's completeness, we are performing a targeted enrichment phase using the OMDB API. This process focuses on retrieving missing Plot summaries for movies that were not covered in the initial import.

In [ ]:
##api_key = API_KEY

def get_plot_from_omdb(row):
    movie_title = row['primaryTitle']
    current_plot = row['Plot']

    if pd.notna(current_plot) and current_plot != "N/A" and len(str(current_plot)) > 10:
        return current_plot

    url = f"http://www.omdbapi.com/?t={movie_title}&apikey={api_key}&plot=full"

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            if data.get('Response') == 'True':
                new_plot = data.get('Plot')
                if new_plot and new_plot != "N/A":
                    return new_plot
        return "N/A"
    except:
        return "N/A"

df_final['Plot'] = df_final.apply(get_plot_from_omdb, axis=1)

display(df_final)

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,Box_Office,Plot
0,tt0002605,The Adventures of Kathlyn,1913,300,[Adventure],5.4,52,"[nm0931031, nm0165134, nm0139356, nm0571186, n...",N/A,United States,NaN,NaN,The daughter of an adventurer in India is kidn...
1,tt0008816,Adam Bede,1918,60,"[Crime, Drama]",NaN,<NA>,"[nm0930154, nm0167087, nm0519307, nm0024706, n...",Silent,United Kingdom,NaN,NaN,A squire's grandson saves a farmer's niece fro...
2,tt0010953,Algol: Tragedy of Power,1920,99,"[Fantasy, Sci-Fi]",6.3,339,"[nm0417837, nm0332024, nm0772300, nm0707765, n...",Silent,Weimar Republic,NaN,NaN,"The story follows the life of Robert Herne, wh..."
3,tt0011908,Adventures of Tarzan,1921,73,"[Action, Adventure]",5.5,187,"[nm0511104, nm0521120, nm0671501, nm0926386, n...",Hindi,N/A,NaN,NaN,This is the popular story of Tarzan retold in ...
4,tt0011909,The Affairs of Anatol,1921,117,"[Comedy, Drama]",6.6,1428,"[nm0717468, nm0841797, nm0223296, nm0199841, n...",Silent,United States,0.176508,1.2,"Socialite Anatol Spencer, finding his relation..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9505,tt9876994,Aamako Man,2018,60,"[Drama, Family]",NaN,<NA>,"[nm10514438, nm10521133, nm10521134, nm1052113...",N/A,N/A,NaN,NaN,N/A
9506,tt9883042,Aakashaganga II,2019,142,[Horror],2.1,316,"[nm0471447, nm11618001, nm9355289, nm5465692, ...",N/A,N/A,NaN,NaN,N/A
9507,tt9900180,Aavahayami,2017,97,[Mystery],8.3,14,"[nm10531566, nm10533868, nm10531569, nm7993373...",N/A,N/A,NaN,NaN,N/A
9508,tt9904552,All for the Money,2019,107,[Comedy],5.4,32,"[nm2113994, nm5711066, nm9295372, nm1693834, n...",N/A,N/A,NaN,NaN,N/A


## Let's evaluate how much this step contributed to our dataset.

In [ ]:
print(f"Number of missing plots: {len(df_final[df_final['Plot'] == 'N/A'])}")

Number of missing plots: 5430


## We are now verifying the data types for each feature within the dataset.

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9510 entries, 0 to 9509
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           9510 non-null   object 
 1   primaryTitle     9510 non-null   object 
 2   startYear        9510 non-null   int32  
 3   runtimeMinutes   9510 non-null   int32  
 4   genres           8980 non-null   object 
 5   averageRating    6785 non-null   float64
 6   numVotes         6785 non-null   Int64  
 7   lead_actors_ids  8049 non-null   object 
 8   Language         9510 non-null   object 
 9   Country          9510 non-null   object 
 10  Budget           487 non-null    float64
 11  Box_Office       640 non-null    float64
 12  Plot             9510 non-null   object 
dtypes: Int64(1), float64(3), int32(2), object(7)
memory usage: 901.0+ KB


## We are now calculating the number of missing values across the entire dataset to assess data quality.

In [ ]:
df_final_cleaned = df_final.replace(['N/A', '', 'None', 'null', ' ', 'nan'], np.nan)

missing_count = df_final_cleaned.isnull().sum()
missing_percentage = (missing_count / len(df_final_cleaned)) * 100

missing_summary = pd.DataFrame({'Column Name': missing_count.index,  'Missing Values': missing_count.values, 'Percentage (%)': missing_percentage.values})

missing_summary = missing_summary.sort_values(by='Missing Values', ascending=False).reset_index(drop=True)

print("Revised Missing Values Analysis (including 'N/A' and empty strings):")
display(missing_summary)

Revised Missing Values Analysis (including 'N/A' and empty strings):


,Column Name,Missing Values,Percentage (%)
0,Budget,9023,94.879075
1,Box_Office,8870,93.270242
2,Country,7188,75.583596
3,Language,6903,72.586751
4,Plot,5430,57.097792
5,averageRating,2725,28.654048
6,numVotes,2725,28.654048
7,lead_actors_ids,1461,15.362776
8,genres,530,5.573081
9,tconst,0,0.000000


## We will select 10 random movies and perform a manual check to verify the integrity and accuracy of the values.

In [ ]:
if 'df_final' in locals() or 'df_final' in globals():
    if not df_final.empty:
        display(df_final.sample(min(10, len(df_final))))
    else:
        print("The DataFrame (df_final) is empty.")
else:
    print("The variable 'df_final' has not been defined yet.")

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,Box_Office,Plot
2184,tt0273053,Aakhosham,1998,132,NaN,NaN,<NA>,"[nm0044183, nm1441726, nm1124690, nm0419677, n...",N/A,N/A,NaN,NaN,"Watch the full movie, Aaghosham, only on Eros ..."
4062,tt13387634,Akilan,2012,122,[Drama],NaN,<NA>,"[nm2128968, nm1007581, nm12031528, nm12031527]",N/A,N/A,NaN,NaN,N/A
8163,tt5076822,Alor Michil,1974,130,"[Drama, History, War]",8.7,99,"[nm6150259, nm2594295, nm0451086, nm0759619, n...",Bengali,N/A,NaN,NaN,The film’s story revolves around a family in t...
3944,tt12992082,Albatross,2022,97,[Drama],4.2,221,"[nm0444534, nm0399126, nm1705116, nm0292021, n...",N/A,N/A,NaN,NaN,N/A
9,tt0017590,An Affair of the Follies,1927,70,"[Drama, Romance]",NaN,<NA>,"[nm0832011, nm0235521, nm0400763, nm0831736, n...",Silent,United States,NaN,NaN,"Young husband Jerry, a clerk, loses his job, a..."
1844,tt0212757,Altri desideri di Karin,1987,83,[Adult],4.7,28,"[nm0775810, nm0035280, nm0457365, nm0124368, n...",N/A,N/A,NaN,NaN,An insatiable woman - upon request of her husb...
6924,tt33043111,Al margen,2024,90,[Documentary],5.8,66,NaN,N/A,N/A,NaN,NaN,N/A
4381,tt14576308,"AC/DC - Black Ice Tour, São Paulo, Brasil 2009...",2009,78,[Music],NaN,<NA>,"[nm0424657, nm0748623, nm0930298, nm0949264, n...",N/A,N/A,NaN,NaN,N/A
4414,tt14760082,Aadi,2005,147,NaN,5.4,11,"[nm2327605, nm0043199, nm5705203, nm3910892, n...",N/A,N/A,NaN,NaN,N/A
9262,tt8784884,Abyakto,2018,85,[Drama],8.2,56,"[nm1300009, nm1137593, nm6166973, nm10217888, ...",Bengali,India,NaN,NaN,Abyakto is a poignant tale of a mother and son...


## The final processed data is exported to a CSV file.

In [ ]:
df_final.to_csv('df_final.csv', index=False)